# Thêm Thư Viện

In [2]:
import pyodbc
import pandas as pd
import numpy as np

# Tạo kết nối

In [3]:
conn_libol = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=192.168.150.6;'  # Địa chỉ IP của SQL Server
    'DATABASE=libol;'         # Tên cơ sở dữ liệu
    'UID=itc;'                # Tên đăng nhập
    'PWD=spkt@2025;'
)
conn_dwh_library = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=192.168.150.6;' # Địa chỉ IP của SQL Server
    'DATABASE=DWH_Lib;' # Tên cơ sở dữ liệu
    'UID=itc;'              # Tên đăng nhập
    'PWD=spkt@2025;'
)

# Đọc data

## Đọc data từ SQL Server

In [4]:
# Hàm đọc dữ liệu từng phần và xử lý lỗi
def fetch_data_in_batches(query_base, connection, batch_size=100):
    offset = 0
    all_data = []  # Lưu tất cả các hàng hợp lệ
    while True:
        query = f"""
        {query_base}
        ORDER BY Tai_lieu_ID
        OFFSET {offset} ROWS FETCH NEXT {batch_size} ROWS ONLY
        """
        try:
            # Đọc dữ liệu batch hiện tại
            df_batch = pd.read_sql(query, connection)
            if df_batch.empty:  # Nếu không còn dữ liệu, dừng vòng lặp
                break
            all_data.append(df_batch)  # Lưu batch hợp lệ
            offset += batch_size  # Tăng offset để đọc batch tiếp theo
        except Exception as e:
            print(f"Lỗi xảy ra khi xử lý batch từ {offset}: {e}")
            offset += batch_size  # Bỏ qua batch bị lỗi và tiếp tục
    # Gộp tất cả các batch thành DataFrame duy nhất
    return pd.concat(all_data, ignore_index=True) if all_data else pd.DataFrame()

In [5]:
query_Tailieu = """
SELECT Tai_lieu_ID,
       Ma_tai_lieu,
       dbo.DecodeUTF8String(Nguoi_nhap_tin) AS Nguoi_nhap_tin,
       dbo.DecodeUTF8String(Nguoi_kiem_tra) AS Nguoi_kiem_tra,
       Nuoc_cung_cap_ID,
       Co_quan_cung_cap_ID,
       Ngay_giao_dich,
       Cap_mo_ta_thu_muc,
       Muc_do_mat,
       Vat_mang_tin_ID,
       Dang_tai_lieu_ID,
       Kieu_ban_ghi,
       Form_ID,
       Leader,
       Anh_bia,
      dbo.DecodeUTF8String(CallNumber) AS CallNumber
  FROM Tai_lieu
"""
df_tailieu = fetch_data_in_batches(query_Tailieu, conn_libol, batch_size=100) # Gọi hàm để lấy dữ liệu
print(df_tailieu)

C:\Users\admin\AppData\Local\Temp\ipykernel_21328\2025791915.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_batch = pd.read_sql(query, connection)


       Tai_lieu_ID  Ma_tai_lieu Nguoi_nhap_tin Nguoi_kiem_tra  \
0                7  SK020000011         DHSPKT           Điền   
1               12  SK020000015         DHSPKT           Điền   
2               26    SKV000004         DHSPKT           Điền   
3               30  SK020000030           luật           Điền   
4               31  SK020000032             vi           Điền   
...            ...          ...            ...            ...   
63163        68641  SK250068651                                 
63164        68642  SK250068652                                 
63165        68643  SK250068653                                 
63166        68644  SK250068654                                 
63167        68645  SK250068655                                 

      Nuoc_cung_cap_ID  Co_quan_cung_cap_ID      Ngay_giao_dich  \
0                 None                  NaN 2007-04-17 07:52:00   
1                 None                  1.0 2007-04-17 07:52:00   
2                 

C:\Users\admin\AppData\Local\Temp\ipykernel_21328\2025791915.py:22: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(all_data, ignore_index=True) if all_data else pd.DataFrame()


### Tạo dataframe backup 

In [6]:
valid_rows = [] # Danh sách lưu các dòng hợp lệ
for index, row in df_tailieu.iterrows(): # Copy từng dòng
    try:
        valid_rows.append(row.copy()) # Thử copy dòng
    except Exception as e:
        print(f"Lỗi khi copy dòng {index}: {e}")
        continue  # Bỏ qua dòng lỗi và tiếp tục

df_tailieu_backup = pd.DataFrame(valid_rows) # Tạo DataFrame mới từ các dòng hợp lệ
print("Số dòng trong df_tailieu:", len(df_tailieu)) # Kiểm tra số lượng dòng
print("Số dòng trong df_tailieu_backup:", len(df_tailieu_backup)) # Kiểm tra số lượng dòng

Số dòng trong df_tailieu: 63168
Số dòng trong df_tailieu_backup: 63168


### [Nếu cần] lấy lại data từ backup

In [7]:
valid_rows = [] # Danh sách lưu các dòng hợp lệ
for index, row in df_tailieu_backup.iterrows(): # Copy từng dòng
    try:
        valid_rows.append(row.copy()) # Thử copy dòng
    except Exception as e:
        print(f"Lỗi khi copy dòng {index}: {e}")
        continue  # Bỏ qua dòng lỗi và tiếp tục

df_tailieu = pd.DataFrame(valid_rows) # Tạo DataFrame mới từ các dòng hợp lệ
print("Số dòng trong df_tailieu_backup:", len(df_tailieu_backup)) # Kiểm tra số lượng dòng
print("Số dòng trong df_tailieu:", len(df_tailieu)) # Kiểm tra số lượng dòng

Số dòng trong df_tailieu_backup: 63168
Số dòng trong df_tailieu: 63168


## Đọc data từ Excel

In [8]:
df_giaotrinh = pd.read_csv("./data_giaotrinh.csv")
df_sudung_giaotrinh = pd.read_csv("./data_sudung_giaotrinh.csv")
print(df_giaotrinh)
print(df_sudung_giaotrinh)

      ID_GiaoTrinh                                             Ten_GT
0               43  Commerce 1:Oxfoxd English for careersent : Stu...
1               44  Intelligent business coursebook :Intermediate ...
2               45                                   Business Matters
3               46                     Communicative Grammar Practice
4               47                                         Let’s talk
...            ...                                                ...
5911          5966  Deep learning for coders with fastai and PyTor...
5912          5967            Pattern Reconition and Machine Learning
5913          5968       Introduction to Machine Learning with Python
5914          5969                Piezoelectric Sensors and Actuators
5915          5970     Sensor and Signal Conditioning, second edition

[5916 rows x 2 columns]
      ID_Mon  ID_GiaoTrinh
0        120          4388
1        120          4859
2        120          4860
3        121          4394


# Xử lý data

## Thêm 1 dòng giả định none

In [9]:
# Tạo DataFrame `new_row` chứa dòng dữ liệu giả định
new_row = pd.DataFrame({
    'Tai_lieu_ID': [0],
    'Ma_tai_lieu': ['(Không xác định)'],
    #'ID_Mon' -- phía dưới xử lý
    'Ngay_giao_dich': ['1024-01-01 00:00:00'],
    'Cap_mo_ta_thu_muc': ['0'],
    'Muc_do_mat': ['0'],
    'Vat_mang_tin_ID': [0],
    'Dang_tai_lieu_ID': [0],
    'Form_ID': [0],
    'Leader': ['(Không xác định)'],
    'CallNumber': ['(Không xác định)']
})
# Thêm dòng dữ liệu giả định vào `df` bằng `pd.concat`
df_tailieu = pd.concat([df_tailieu, new_row], ignore_index=True) # Thêm vào dataframe
df_tailieu = df_tailieu.sort_values(by="Tai_lieu_ID", ascending=True).reset_index(drop=True) # sắp xếp từ nhỏ đến lớn           
print(df_tailieu)

       Tai_lieu_ID       Ma_tai_lieu Nguoi_nhap_tin Nguoi_kiem_tra  \
0                0  (Không xác định)            NaN            NaN   
1                7       SK020000011         DHSPKT           Điền   
2               12       SK020000015         DHSPKT           Điền   
3               26         SKV000004         DHSPKT           Điền   
4               30       SK020000030           luật           Điền   
...            ...               ...            ...            ...   
63164        68641       SK250068651                                 
63165        68642       SK250068652                                 
63166        68643       SK250068653                                 
63167        68644       SK250068654                                 
63168        68645       SK250068655                                 

      Nuoc_cung_cap_ID  Co_quan_cung_cap_ID       Ngay_giao_dich  \
0                  NaN                  NaN  1024-01-01 00:00:00   
1                 None 

## Xử lý ID_Tai_lieu

In [10]:
# Lọc ra các ID_GiaoTrinh chưa có trong df_tailieu
new_rows = df_giaotrinh[~df_giaotrinh['ID_GiaoTrinh'].isin(df_tailieu['Tai_lieu_ID'])]
new_rows = new_rows[['ID_GiaoTrinh']].rename(columns={'ID_GiaoTrinh': 'Tai_lieu_ID'})
df_tailieu = pd.concat([df_tailieu, new_rows], ignore_index=True)
df_tailieu.drop(columns=['Ten_GT_x', 'ID_GiaoTrinh', 'Ten_GT_y'], errors='ignore', inplace=True)

print(f"Số hàng: {df_tailieu.shape[0]}, Số cột: {df_tailieu.shape[1]}")
print("Tên các cột:", df_tailieu.columns.tolist())


Số hàng: 63531, Số cột: 16
Tên các cột: ['Tai_lieu_ID', 'Ma_tai_lieu', 'Nguoi_nhap_tin', 'Nguoi_kiem_tra', 'Nuoc_cung_cap_ID', 'Co_quan_cung_cap_ID', 'Ngay_giao_dich', 'Cap_mo_ta_thu_muc', 'Muc_do_mat', 'Vat_mang_tin_ID', 'Dang_tai_lieu_ID', 'Kieu_ban_ghi', 'Form_ID', 'Leader', 'Anh_bia', 'CallNumber']


## Xử lý data rỗng hoặc " "

In [11]:
df_tailieu = df_tailieu.replace('', None)
df_tailieu = df_tailieu.replace(np.nan, None)
print(df_tailieu)

       Tai_lieu_ID       Ma_tai_lieu Nguoi_nhap_tin Nguoi_kiem_tra  \
0                0  (Không xác định)           None           None   
1                7       SK020000011         DHSPKT           Điền   
2               12       SK020000015         DHSPKT           Điền   
3               26         SKV000004         DHSPKT           Điền   
4               30       SK020000030           luật           Điền   
...            ...               ...            ...            ...   
63526         5880              None           None           None   
63527         5887              None           None           None   
63528         5888              None           None           None   
63529         5896              None           None           None   
63530         5933              None           None           None   

      Nuoc_cung_cap_ID Co_quan_cung_cap_ID       Ngay_giao_dich  \
0                 None                None  1024-01-01 00:00:00   
1                 None   

## Xử lý Ma_Tai_lieu

In [12]:
df_tailieu['Ma_tai_lieu'] = df_tailieu['Ma_tai_lieu'].replace("", None)  # Thay giá trị chuỗi rỗng thành None
df_tailieu['Ma_tai_lieu'] = df_tailieu['Ma_tai_lieu'].fillna("(Không xác định)")
print(df_tailieu[['Ma_tai_lieu']])

            Ma_tai_lieu
0      (Không xác định)
1           SK020000011
2           SK020000015
3             SKV000004
4           SK020000030
...                 ...
63526  (Không xác định)
63527  (Không xác định)
63528  (Không xác định)
63529  (Không xác định)
63530  (Không xác định)

[63531 rows x 1 columns]


## Xử lý Ten_tai_lieu

In [13]:
# Ghép dữ liệu và thêm cột 'Ten_tai_lieu'
df_tailieu = df_tailieu.merge(
    df_giaotrinh[['ID_GiaoTrinh', 'Ten_GT']],
    left_on='Tai_lieu_ID',
    right_on='ID_GiaoTrinh',
    how='left'
)

# Đổi tên cột
df_tailieu.rename(columns={'Ten_GT': 'Ten_tai_lieu'}, inplace=True) # Đổi tên cột
df_tailieu['Ten_tai_lieu'] = df_tailieu['Ten_tai_lieu'].where(df_tailieu['Ten_tai_lieu'].notnull(), None)
df_tailieu.drop('ID_GiaoTrinh', axis=1, inplace=True)
print(df_tailieu[['Tai_lieu_ID', 'Ten_tai_lieu']])


       Tai_lieu_ID                                       Ten_tai_lieu
0                0                                               None
1                7                                               None
2               12                                               None
3               26                                               None
4               30                                               None
...            ...                                                ...
63526         5880            Các văn kiện quốc tế về quyền con người
63527         5887  Luật hợp đồng Việt Nam – Bản án và bình luận b...
63528         5888   Pháp luật hợp đồng- một số vấn đề pháp lý cơ bản
63529         5896                         Giáo trình luật cạnh tranh
63530         5933                Giáo trình Kỹ năng tư vấn pháp luật

[63531 rows x 2 columns]


## Xử lý ID_mon

In [14]:
# Ghép dữ liệu và thêm cột 'ID_Mon'
df_tailieu = df_tailieu.merge(df_sudung_giaotrinh[['ID_GiaoTrinh', 'ID_Mon']],
                                left_on='Tai_lieu_ID',
                                right_on='ID_GiaoTrinh',
                                how='left')

df_tailieu['ID_Mon'] = df_tailieu['ID_Mon'].apply(lambda x: 0 if pd.isna(x) else x) # Xử lý giá trị thiếu bằng 0
df_tailieu.drop('ID_GiaoTrinh', axis=1, inplace=True) # Xóa cột ID_GiaoTrinh thừa
print(df_tailieu['ID_Mon'])

0           0.0
1           0.0
2           0.0
3           0.0
4           0.0
          ...  
63526    1999.0
63527    1992.0
63528    1992.0
63529    1998.0
63530    1959.0
Name: ID_Mon, Length: 63531, dtype: float64


## Xử lý Ngay_giao_dich

In [15]:
query_date = "SELECT Date_key FROM olap.DIM_Date"
df_date = pd.read_sql(query_date, conn_dwh_library)
date_ids = set(df_date['Date_key'])
# chuyển date về dang int 
# kiểm tra nhưng ngày đó có tồn tại trong date_key của bảng DIM_date hay không ?
df_tailieu['Ngay_giao_dich'] = pd.to_datetime(df_tailieu['Ngay_giao_dich'], errors='coerce')
df_tailieu['Ngay_giao_dich'] = df_tailieu['Ngay_giao_dich'].apply(lambda x: int(x.strftime('%Y%m%d')) if pd.notna(x) and int(x.strftime('%Y%m%d')) in date_ids else 0)
print(df_tailieu[['Ngay_giao_dich']])

C:\Users\admin\AppData\Local\Temp\ipykernel_21328\3478724221.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_date = pd.read_sql(query_date, conn_dwh_library)
C:\Users\admin\AppData\Local\Temp\ipykernel_21328\3478724221.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_tailieu['Ngay_giao_dich'] = pd.to_datetime(df_tailieu['Ngay_giao_dich'], errors='coerce')


       Ngay_giao_dich
0                   0
1            20070417
2            20070417
3            20070417
4            20070417
...               ...
63526               0
63527               0
63528               0
63529               0
63530               0

[63531 rows x 1 columns]


## Xử lý ID_quoc_gia

In [16]:
query_Quocgia = "SELECT ID_quoc_gia FROM olap.DIM_Quoc_gia"
df_quocgia = pd.read_sql(query_Quocgia, conn_dwh_library)
quocgia_ids = set(df_quocgia['ID_quoc_gia'])
# chuyển date về dang int 
# kiểm tra nhưng ngày đó có tồn tại trong ID_quoc_gia của bảng DIM_Quoc_gia hay không ?
df_tailieu['Nuoc_cung_cap_ID'] = df_tailieu['Nuoc_cung_cap_ID'].apply(lambda x: x if pd.notna(x) and x in quocgia_ids else 0)
print(df_tailieu[['Nuoc_cung_cap_ID']])

       Nuoc_cung_cap_ID
0                     0
1                     0
2                     0
3                     0
4                     0
...                 ...
63526                 0
63527                 0
63528                 0
63529                 0
63530                 0

[63531 rows x 1 columns]


C:\Users\admin\AppData\Local\Temp\ipykernel_21328\3230730724.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_quocgia = pd.read_sql(query_Quocgia, conn_dwh_library)


## Xử lý Cap_mo_ta_thu_muc

In [17]:
df_tailieu['Cap_mo_ta_thu_muc'] = df_tailieu['Cap_mo_ta_thu_muc'].apply(lambda x: '0' if pd.isna(x) or x == '' else x)
print(df_tailieu['Cap_mo_ta_thu_muc'])

0        0
1        m
2        m
3        m
4        m
        ..
63526    0
63527    0
63528    0
63529    0
63530    0
Name: Cap_mo_ta_thu_muc, Length: 63531, dtype: object


## Xử lý Muc_do_mat

In [18]:
df_tailieu['Muc_do_mat'] = df_tailieu['Muc_do_mat'].apply(lambda x: '0' if pd.isna(x) or x == '' else x)
print(df_tailieu['Muc_do_mat'])

0        0
1        A
2        A
3        A
4        A
        ..
63526    0
63527    0
63528    0
63529    0
63530    0
Name: Muc_do_mat, Length: 63531, dtype: object


## Xử lý ID_vat_mang_tin

In [19]:
query_Vatmangtin = "SELECT ID_vat_mang_tin FROM olap.DIM_Vat_mang_tin"
df_vatmangtin = pd.read_sql(query_Vatmangtin, conn_dwh_library)
vatmangtin_ids = set(df_vatmangtin['ID_vat_mang_tin'])
# chuyển date về dang int
# kiểm tra nhưng ngày đó có tồn tại trong ID_vat_mang_tin của bảng DIM_Vat_mang_tin hay không ?
df_tailieu['Vat_mang_tin_ID'] = df_tailieu['Vat_mang_tin_ID'].apply(lambda x: x if pd.notna(x) and x in vatmangtin_ids else 0)
print(df_tailieu[['Vat_mang_tin_ID']])

       Vat_mang_tin_ID
0                  0.0
1                  3.0
2                  3.0
3                  3.0
4                  3.0
...                ...
63526              0.0
63527              0.0
63528              0.0
63529              0.0
63530              0.0

[63531 rows x 1 columns]


C:\Users\admin\AppData\Local\Temp\ipykernel_21328\1452113009.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_vatmangtin = pd.read_sql(query_Vatmangtin, conn_dwh_library)


## Xử lý ID_dang_tai_lieu

In [20]:
query_Dangtailieu = "SELECT ID_dang_tai_lieu FROM olap.DIM_Dang_tai_lieu"
df_dangtailieu = pd.read_sql(query_Dangtailieu, conn_dwh_library)
dangtailieu_ids = set(df_dangtailieu['ID_dang_tai_lieu'])
# chuyển date về dang int
# kiểm tra nhưng ngày đó có tồn tại trong ID_dang_tai_lieu của bảng DIM_Dang_tai_lieu hay không ?
df_tailieu['Dang_tai_lieu_ID'] = df_tailieu['Dang_tai_lieu_ID'].apply(lambda x: x if pd.notna(x) and x in dangtailieu_ids else 0)
print(df_tailieu[['Dang_tai_lieu_ID']])

       Dang_tai_lieu_ID
0                   0.0
1                   1.0
2                   1.0
3                   1.0
4                   1.0
...                 ...
63526               0.0
63527               0.0
63528               0.0
63529               0.0
63530               0.0

[63531 rows x 1 columns]


C:\Users\admin\AppData\Local\Temp\ipykernel_21328\1178039827.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_dangtailieu = pd.read_sql(query_Dangtailieu, conn_dwh_library)


## Xử lý ID_form

In [21]:
query_form = "SELECT ID_form FROM olap.DIM_Ten_form"
df_tenform = pd.read_sql(query_form, conn_dwh_library)
tenform_ids = set(df_tenform['ID_form'])
# chuyển date về dang int
# kiểm tra nhưng ngày đó có tồn tại trong ID_dang_tai_lieu của bảng DIM_Dang_tai_lieu hay không ?
df_tailieu['Form_ID'] = df_tailieu['Form_ID'].apply(lambda x: x if pd.notna(x) and x in tenform_ids else 0)
print(df_tailieu[['Form_ID']])

       Form_ID
0          0.0
1         83.0
2         83.0
3         83.0
4         83.0
...        ...
63526      0.0
63527      0.0
63528      0.0
63529      0.0
63530      0.0

[63531 rows x 1 columns]


C:\Users\admin\AppData\Local\Temp\ipykernel_21328\3707907086.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_tenform = pd.read_sql(query_form, conn_dwh_library)


## Load data

### [Nếu cần] Clear bảng

In [22]:
cursor = conn_dwh_library.cursor()
truncate_query = "DELETE FROM olap.DIM_Tai_lieu"
cursor.execute(truncate_query)
conn_dwh_library.commit()
cursor.close()

In [23]:
for col in df_tailieu.columns:
    print(f"Cột: {col} - Độ dài lớn nhất: {df_tailieu[col].astype(str).apply(len).max()}")


Cột: Tai_lieu_ID - Độ dài lớn nhất: 5
Cột: Ma_tai_lieu - Độ dài lớn nhất: 16
Cột: Nguoi_nhap_tin - Độ dài lớn nhất: 23
Cột: Nguoi_kiem_tra - Độ dài lớn nhất: 22
Cột: Nuoc_cung_cap_ID - Độ dài lớn nhất: 1
Cột: Co_quan_cung_cap_ID - Độ dài lớn nhất: 4
Cột: Ngay_giao_dich - Độ dài lớn nhất: 8
Cột: Cap_mo_ta_thu_muc - Độ dài lớn nhất: 1
Cột: Muc_do_mat - Độ dài lớn nhất: 1
Cột: Vat_mang_tin_ID - Độ dài lớn nhất: 4
Cột: Dang_tai_lieu_ID - Độ dài lớn nhất: 4
Cột: Kieu_ban_ghi - Độ dài lớn nhất: 4
Cột: Form_ID - Độ dài lớn nhất: 4
Cột: Leader - Độ dài lớn nhất: 24
Cột: Anh_bia - Độ dài lớn nhất: 41
Cột: CallNumber - Độ dài lớn nhất: 34
Cột: Ten_tai_lieu - Độ dài lớn nhất: 402
Cột: ID_Mon - Độ dài lớn nhất: 6


### Load data vào bảng Dim

In [24]:
# Tạo cursor để thao tác với cơ sở dữ liệu
cursor_dwh = conn_dwh_library.cursor()

# Chuẩn bị câu lệnh chèn dữ liệu
insert_query = """
                INSERT INTO olap.DIM_Tai_lieu (
                    ID_tai_lieu, Ma_tai_lieu, Ten_tai_lieu, 
                    Nguoi_nhap_tin, Nguoi_kiem_tra, 
                    ID_mon, ID_quoc_gia, ID_co_quan_cung_cap,
                    Ngay_giao_dich,
                    Cap_mo_ta_thu_muc, Muc_do_mat, 
                    ID_vat_mang_tin, ID_dang_tai_lieu,
                    Kieu_ban_ghi, ID_form, 
                    Leader, CallNumber
                ) 
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
               """
# Chuyển đổi dữ liệu từ DataFrame thành danh sách các tuple để chèn
data_to_insert = [
    (
        row['Tai_lieu_ID'], row['Ma_tai_lieu'], row['Ten_tai_lieu'],
        row['Nguoi_nhap_tin'], row['Nguoi_kiem_tra'],
        row['ID_Mon'], row['Nuoc_cung_cap_ID'], row['Co_quan_cung_cap_ID'],
        row['Ngay_giao_dich'], 
        row['Cap_mo_ta_thu_muc'], row['Muc_do_mat'],
        row['Vat_mang_tin_ID'], row['Dang_tai_lieu_ID'], 
        row['Kieu_ban_ghi'], row['Form_ID'],
        row['Leader'], row['CallNumber'], 
    )
    for index, row in df_tailieu.iterrows()
]
# Sử dụng executemany để chèn dữ liệu cùng lúc
cursor_dwh.executemany(insert_query, data_to_insert)
# Commit thay đổi
conn_dwh_library.commit()
# Đóng cursor và kết nối
cursor_dwh.close()
conn_dwh_library.close()